# Paradata Anomaly & Feature Analysis

This notebook provides a robust, visual analysis of the features extracted by the paradata pipeline (`features_output.csv`).

---

## 1. Required CSV Format

Before running the pipeline, your raw audit files located in the `data/` folder must be in **CSV format**.

The pipeline is extremely robust and will try to automatically identify your columns (handling both ODK and SurveyCTO formats). However, for the math to work properly, your file **MUST** contain at least some of the following columns:

- `interview_id`: Unique identifier for the respondent.
- `node` or `question_id`: The name of the question/screen.
- `event` or `event_type`: The action occurring (e.g., `Question`, `backward`, `pause`).
- `start` or `start_time`: Time the question began.
- `end` or `end_time`: Time the question concluded.

### How does the pipeline handle broken datasets?
If you accidentally upload a broken dataset (like your example: `201,Q1,0,-5,answer`), the system is designed to **not crash**. Instead:
1. **Missing Data**: If timestamps are totally missing or unreadable, the script sets that row's duration to `NaN` (Not a Number) without halting.
2. **Negative Time**: If a start time is later than an end time (resulting in negative duration), the parser physically deletes the negative number and replaces it with `NaN` so it doesn't violently skew the mean response times negatively.
3. **Completely Corrupt Files**: If a file is missing _all_ the required columns, the pipeline throws a clear `VALIDATION ERROR` specifying exactly which headers it requires, rather than throwing a cryptic internal Python crash.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Set aesthetic style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Load the Extracted Features
We load the most recent `features_output.csv` generated by `run_pipeline.py`.

In [ ]:
import glob

# Find the newest output file
output_files = glob.glob("../output/features_output_*.csv")
if not output_files:
    print("No generic timestamped files found. Falling back to default...")
    output_file = "../output/features_output.csv"
else:
    output_file = max(output_files, key=os.path.getctime)

print(f"Loading: {output_file}")
df = pd.read_csv(output_file)

display(df.head())

## 3. Response Time Distribution
Visualizing the mean response times allows us to spot the 'speeders'.

In [ ]:
plt.figure(figsize=(10, 5))

# Color specific interviews based on anomaly status if it exists
if 'is_anomalous' in df.columns:
    colors = df['is_anomalous'].map({1: 'red', 0: 'blue'})
    sns.scatterplot(data=df, x='mean_response_time_ms', y='total_duration_ms', 
                    hue='is_anomalous', palette={0: 'blue', 1: 'red'}, size='straightlining_count', sizes=(50, 400))
else:
    sns.scatterplot(data=df, x='mean_response_time_ms', y='total_duration_ms', size='straightlining_count', sizes=(50, 400))

plt.title('Interview Mean Response Times vs Total Duration\n(Bubble size = straightlining count)')
plt.xlabel('Mean Response Time (ms)')
plt.ylabel('Total Interview Duration (ms)')
plt.annotate('Fast & Constant\nStraightlining!', 
             xy=(df['mean_response_time_ms'].min(), df['total_duration_ms'].min()), 
             xycoords='data', xytext=(50, 50), textcoords='offset points',
             arrowprops=dict(facecolor='red', shrink=0.05))

plt.show()

## 4. Feature Correlations
Are enumerators who speed through surveys (low Speed Index) also the ones creating low entropy (robot-like uniform answering)?

In [ ]:
# Isolate numerical features for the correlation matrix
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", 
            vmin=-1, vmax=1, square=True, linewidths=.5)
plt.title('Correlation Matrix of Paradata Features')
plt.show()

## 5. Anomaly Detection Focus
Let's explicitly check the interviews the system previously flagged as anomalous via Z-Scores.

In [ ]:
if 'is_anomalous' in df.columns:
    anomalies = df[df['is_anomalous'] == 1]
    print(f"Found {len(anomalies)} heavily anomalous interviews.")
    display(anomalies[['interview_id', 'speed_index', 'straightlining_count', 'z_score_duration', 'z_score_straightlining']])
else:
    print("No anomaly z-scores were calculated during the last pipeline run.")